# Market-making markouts: event-log wrangling lab
**Synthetic training exercise · 60–120 minutes · beginner-to-intermediate pandas**

You are given **one messy event CSV** containing BTC/ETH market quotes, fills, cancel decisions/sends/rejects, and acknowledgements. Your job is to transform it into a fill-level table and calculate markouts. No firm data is needed.

**Focus:** 70% cleaning, slicing, joins and timestamps; 30% markout analysis. **Important:** all rows are invented; no live or firm data. Work through each `TODO` cell before opening the separate answer key.

## Data dictionary and what is intentionally messy

- `event_id`: unique event identifier, except deliberately duplicated raw rows.
- `event_type`: `QUOTE`, `NEW_ACK`, `FILL`, `CANCEL_DECISION`, `CANCEL_SEND`, `CANCEL_REJECT`, `CANCEL_ACK`.
- `recv_time`, `exch_time`: timestamp strings; exchange time is absent for internal events. **Do not assume different clocks are synchronized in real trading data.**
- `symbol`: BTC-PERP or ETH-PERP, occasionally with whitespace/lowercase.
- `TTOrderID`, `cl_ord_id`: order identifiers (blank on quotes).
- `side`, `price`, `qty`: meaningful mostly on fills. `mid_price`: meaningful only on quotes.
- Rows are shuffled, some duplicated, some quote mids missing. Nulls in unrelated columns are *expected*, not universally bad.

**Your target output:** one row per BTC-PERP fill with group, four signed markouts and a short cost summary.

**Markout convention:** maker BUY → +1; maker SELL → −1. Signed markout (bps) = `side_sign * (future_mid - fill_price) / fill_price * 1e4`. This is a mark-to-market proxy, not realised strategy PnL.

## 0 · Load & inspect (5 min)
Put `synthetic_trading_events.csv` in the same directory as this notebook. Run the starter cell, then inspect `raw.dtypes`, `.isna().mean()`, and `event_type.value_counts()`. What nulls are structurally expected?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('synthetic_trading_events.csv')
raw = pd.read_csv(DATA)
print('Raw:', raw.shape)
display(raw.head(5))

## 1 · Clean the raw log (15–20 min)
**Do:** strip/uppercase `symbol`, `side`, `event_type`; parse UTC timestamps with `pd.to_datetime`; remove duplicates by `event_id`; sort events in time order; create a `session_date` string.

**Check:** `event_id` unique, timestamps valid, only two clean symbols. Avoid `dropna()` on the entire table.

In [ ]:
# TODO: create `events` from raw
# events = ...


## 2 · Slice to BTC and construct clean quote/fill tables (10–15 min)
**Do:** use `.loc` and explicit column selection. `quotes` needs `recv_ts`, `symbol`, `session_date`, `mid_price`; `fills` needs order ID, fill timestamp, side, price, quantity. Convert numeric columns, rename to `fill_ts` / `fill_price`. Drop missing mids only from quotes.

**Check:** each fill event remains a single row.

In [ ]:
# TODO: create `btc`, `quotes`, `fills`


## 3 · Join decisions, sends & rejects to fills (15–20 min)
**Do:** slice cancellation events to separate small tables and left-join them onto fills by `TTOrderID`; use `validate='many_to_one'`. Classify each fill as:

- `ordinary`: no cancel reject
- `decided_too_late`: cancel decision after the fill
- `lost_to_latency`: cancel sent before the fill, but still rejected

Use an explicit `review` fallback rather than silently assuming every row fits. **Caveat:** these labels are valid for this simplified synthetic dataset; actual exchange event ordering needs stronger timestamp and matching evidence.

In [ ]:
# TODO: join and classify fills


## 4 · Timestamp joins with `merge_asof` (20–25 min)
**Do:** first match each fill to the most recent quote (`direction='backward'`). Then for each horizon (100ms, 1s, 10s, 60s), construct `target_ts = fill_ts + horizon` and match the **first quote at or after** that target. Use `by=['symbol','session_date']` and a `250ms` tolerance.

**Trap:** sort both frames by their actual timestamp keys globally, even when joining `by=` symbol/day. Verify matched row counts and null rates.

**Reason:** ordinary exact `merge` fails if quote timestamps and fill timestamps differ by milliseconds.

In [ ]:
# TODO: baseline merge_asof and 4 forward-horizon merge_asof joins


## 5 · Calculate markouts, aggregate and plot (15–20 min)
**Do:** add `side_sign` and signed markouts in bps and USD for all four horizons. Summarise per `fill_group`, estimate the **difference** between ordinary and post-cancel average markout, multiply by late fills per day. Make one bar chart.

**Interpretation:** distinguish markout from actual realised PnL. A late/ordinary difference is descriptive, not automatically causal.

In [ ]:
# TODO: calculate markouts and group cost estimates


## 6 · Stretch: a simple day-level confidence interval (optional, 15 min)
Create a daily late-fill cost series; bootstrap by **day**, not by fill. With only six synthetic trading days, treat the CI as an exercise rather than a precise inference.

In [ ]:
# OPTIONAL TODO: day-level bootstrap


## Self-check & practical pandas toolbox

Run these after completing the relevant TODOs. Uncomment assertions as you reach each milestone.

```python
assert events['event_id'].is_unique
assert set(events['symbol'].dropna().unique()) == {'BTC-PERP', 'ETH-PERP'}
assert fills['TTOrderID'].is_unique
assert fills['fill_group'].ne('review').all()
assert enriched['mid_60s'].notna().mean() > 0.98
```

Useful patterns: `df.loc[mask, cols]`, `df.drop_duplicates(subset=[...])`, `df.merge(other, on='TTOrderID', how='left', validate='many_to_one')`, `pd.merge_asof(...)`, `df.groupby(...).agg(...)`.

**Follow-on:** Try a `hedge_fill` dataset next, matching first-leg fills to hedge fills, then use the same timestamp-join pattern for slippage.

**Markouts are conventions, not facts:** note fill-side sign, execution price, quote freshness, venue, holding horizon, and clock synchronisation in every memo.